In [1]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import lightgbm as lgb

from sklearn.metrics import accuracy_score, roc_auc_score

from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    KFold,
    StratifiedKFold,
    GroupKFold,
    StratifiedGroupKFold,
)

In [ ]:
df = pd.read_csv("healthcare-dataset-stroke-data.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [11]:
df["ever_married"] = df["ever_married"].replace("Yes", True).replace("No", False)
df["gender"] = df["gender"].astype("category")
df["smoking_status"] = df["smoking_status"].astype("category")
df["Residence_type"] = df["Residence_type"].astype("category")
df["work_type"] = df["work_type"].astype("category")



In [13]:
df["doctor"] = np.random.randint(0,8,len(df))
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke,doctor
0,9046,Male,67.0,0,1,True,Private,Urban,228.69,36.6,formerly smoked,1,1
1,51676,Female,61.0,0,0,True,Self-employed,Rural,202.21,NaN,never smoked,1,5
2,31112,Male,80.0,0,1,True,Private,Rural,105.92,32.5,never smoked,1,7
3,60182,Female,49.0,0,0,True,Private,Urban,171.23,34.4,smokes,1,1
4,1665,Female,79.0,1,0,True,Self-employed,Rural,174.12,24.0,never smoked,1,2


In [23]:
df["gender"].unique()

['Male', 'Female', 'Other']
Categories (3, object): ['Female', 'Male', 'Other']

In [25]:
holdout_ids = df.sample(n=500, random_state=519).index
train = (
    df.loc[~df.index.isin(holdout_ids)]
    .sample(frac=1, random_state=529)
    .sort_values("doctor")
    .reset_index(drop=True)

)
holdout = (
    df.loc[df.index.isin(holdout_ids)]
    .sample(frac=1, random_state=529)
    .sort_values("doctor")
    .reset_index(drop=True)
    )

return train, holdout

SyntaxError: 'return' outside function (3033418199.py, line 16)

In [26]:
def get_prep_data():
    data = pd.read_csv("healthcare-dataset-stroke-data.csv")
    
    data["ever_married"] = data["ever_married"].replace("Yes", True).replace("No", False)
    data["gender"] = data["gender"].astype("category")
    data["smoking_status"] = data["smoking_status"].astype("category")
    data["Residence_type"] = data["Residence_type"].astype("category")
    data["work_type"] = data["work_type"].astype("category")
    data["doctor"] = np.random.randint(0, 8, size=len(data))

    holdout_ids = data.sample(n=500, random_state=529).index

    train = (
        data.loc[~data.index.isin(holdout_ids)]
        .sample(frac=1, random_state=529)
        .sort_values("doctor")
        .reset_index(drop=True)
    )
    holdout = (
        data.loc[data.index.isin(holdout_ids)]
        .sample(frac=1, random_state=529)
        .sort_values("doctor")
        .reset_index(drop=True)
    )

    return train, holdout

train, holdout = get_prep_data()

In [30]:
holdout_ids = df.sample(n=500, random_state=69).index

train = (
    df.loc[~df.index.isin(holdout_ids)]
    .sample(frac=1, random_state=69)
    .sort_values("doctor")
    .reset_index(drop=True)

)

holdout = (
    df.loc[df.index.isin(holdout_ids)]
    .sample(frac=1, random_state=69)
    .sort_values("doctor")
    .reset_index(drop=True)
)




In [31]:
def get_prep_data():
    data = pd.read_csv("healthcare-dataset-stroke-data.csv")
    
    data["ever_married"] = data["ever_married"].replace("Yes", True).replace("No", False)
    data["gender"] = data["gender"].astype("category")
    data["smoking_status"] = data["smoking_status"].astype("category")
    data["Residence_type"] = data["Residence_type"].astype("category")
    data["work_type"] = data["work_type"].astype("category")
    data["doctor"] = np.random.randint(0, 8, size=len(data))

    holdout_ids = data.sample(n=500, random_state=529).index

    train = (
        data.loc[~data.index.isin(holdout_ids)]
        .sample(frac=1, random_state=529)
        .sort_values("doctor")
        .reset_index(drop=True)
    )
    holdout = (
        data.loc[data.index.isin(holdout_ids)]
        .sample(frac=1, random_state=529)
        .sort_values("doctor")
        .reset_index(drop=True)
    )

    return train, holdout

train, holdout = get_prep_data()

In [34]:
train.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke', 'doctor'],
      dtype='object')

In [36]:
def get_X_y(train):
    FEATURES= [
        "id",
        "gender",
        "age",
        "hypertension"
        "heart_disease"
        "ever_married"
        "work_type"
        "Residence_type"
        "avg_glucose_level"
        "bmi"
        "smoking_status"

    ]
    TARGET = "stroke"
    GROUP="doctor"
    X=train[FEATURES]
    Y=train[TARGET]
    groups=train[GROUP]
    return X, Y, groups



In [ ]:
clf = lgb.LGBMClassifier(n_estimators=100)
clf.fit(X,Y)
pred =clf.predict(X)
pred_prob= clf.predict[:,-1]
